# Unstructured Data Summative -  Web scraping and Sentiment analysis on Trustpilot Reviews



---



---



*This is now a defunct piece of code left here for posterity.
I had made an attempt to use  selenium drivers that would work on colab specifcally, but due to numerous errors that kept arising while running the code, i settled on using the BeautifulSoup packages instead.*

    %pip install google-colab-selenium
    pip install selenium webdriver-manager

    import google_colab_selenium as gs

    driver = gs.Chrome()
    driver.quit()


    !pip install selenium webdriver-manager
    !apt-get update # to update ubuntu to correctly run apt install
    !apt install chromium-chromedriver
    !cp /usr/lib/chromium-browser/chromedriver /usr/bin

    import time
    import json
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.common.keys import Keys
    import sys
    import google_colab_selenium as gs


    sys.path.insert(0,'/usr/lib/chromium-browser/chromedriver')
    Trustpilot URL (Example: Amazon UK)
    URL = "https://www.trustpilot.com/review/www.amazon.co.uk"

    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument('--headless')
    chrome_options.add_argument('--no-sandbox')
    chrome_options.add_argument('--disable-dev-shm-usage')
    driver = gs.Chrome(options=chrome_options)



    def get_reviews(url, max_scrolls=3):
        driver.get(url)
        time.sleep(3)  # Allow time for page to load

        # Scroll down to load more reviews
        for _ in range(max_scrolls):
            driver.find_element(By.TAG_NAME, "body").send_keys(Keys.END)
            time.sleep(3)  # Wait for new reviews to load

        reviews = []
        review_containers = driver.find_elements(By.CSS_SELECTOR, "section.styles_reviewCard__hcAvl")

        for review in review_containers:
            try:
                title = review.find_element(By.CSS_SELECTOR, "h2.typography_heading-s__f7029").text.strip()
                content = review.find_element(By.CSS_SELECTOR, "p.typography_body-l__KUYFJ").text.strip()
                rating = review.find_element(By.CSS_SELECTOR, "div.star-rating_starRating__4rrcf").get_attribute("aria-label")
                date = review.find_element(By.TAG_NAME, "time").get_attribute("datetime")

                reviews.append({
                    "title": title,
                    "content": content,
                    "rating": rating,
                    "date": date
                })
            except Exception as e:
                print(f"Error extracting review: {e}")
                continue  # Skip if any element is missing

        return reviews

    if __name__ == "__main__":
        scraped_reviews = get_reviews(URL)
        driver.quit()  # Close the browser

        if scraped_reviews:
            print(json.dumps(scraped_reviews, indent=4, ensure_ascii=False))
        else:
            print("No reviews found.")
    """

---

---



---



---



# 1. Importing Beautiful Soup and conducting the web scraping

In [ ]:
!pip install transformers
!pip install bertopic

import pandas as pd
from bs4 import BeautifulSoup
import requests, csv
import logging
import unittest
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt
import re
import nltk
nltk.download('stopwords')
nltk.download('wordnet')
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
nltk.download('vader_lexicon')
from nltk.sentiment.vader import SentimentIntensityAnalyzer
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from bertopic import BERTopic
import seaborn as sns
from collections import defaultdict

from sklearn.metrics.pairwise import cosine_similarity  # Use cosine similarity
from scipy.cluster.hierarchy import linkage


#Web Scraping and Data Extraction from Trustpilot

This Python code utilizes the requests and BeautifulSoup libraries to perform web scraping, extracting customer reviews from Trustpilot for BT (Richardson, 2007). It incorporates logging for progress tracking and error handling, and utilizes the unittest framework for validation.

###Key Features:


1.   **Targeted Scraping:** The scrape_reviews function fetches HTML content from a given Trustpilot URL and iterates through specified page numbers (from_page to to_page). It employs CSS selectors within BeautifulSoup to precisely locate and extract review elements, including title, date, rating, and text, storing them in a structured list of dictionaries.
2.   **Robustness and Error Handling:** The code includes try-except blocks to handle potential network errors (requests.exceptions.RequestException) and unexpected exceptions during the scraping process. Comprehensive logging using logging.basicConfig records progress, errors, and exceptions, facilitating debugging and monitoring.
3. **Automated Testing:** A TestScraping class leverages unittest to verify the functionality of the scrape_reviews function. It asserts that at least one review is scraped, ensuring the scraping logic's effectiveness and alerting to potential issues caused by website changes.

4. **Data Persistence:** The extracted review data is saved into a CSV file (zzz_my_result.csv) using the csv library. This enables data persistence for further analysis, visualization, and integration with other data science workflows

###Data Science Relevance:
This code provides a foundational step in a typical data science pipeline involving unstructured data:

* **Data Acquisition:** It enables the collection of raw customer feedback from a
real-world source, Trustpilot, providing valuable insights into customer sentiment and experiences.
* **Data Preprocessing:** The extracted data can be further cleaned, processed, and transformed using NLP techniques for sentiment analysis, topic modeling, or feature engineering.
* **Model Training and Evaluation:** The structured review data can be used to train machine learning models for tasks like sentiment classification or to build recommender systems.
* **Stakeholder Communication:** The generated CSV file facilitates data sharing and collaboration with stakeholders, enabling them to understand customer feedback trends and make data-driven decisions.

Libraries Used:
* requests: for fetching HTML content from web pages.
* BeautifulSoup: for parsing HTML and extracting data.
* logging: for recording progress and errors.
* unittest: for automated testing.
* csv: for saving data to a CSV file.

In [ ]:
# Configure logging
logging.basicConfig(filename='scraping_log.txt', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

def scrape_reviews(url, from_page, to_page):
    """
    Scrapes reviews from Trustpilot for a given URL and page range.

    Args:
        url (str): The URL of the Trustpilot review page.
        from_page (int): The starting page number.
        to_page (int): The ending page number.

    Returns:
        list: A list of dictionaries, where each dictionary represents a review with keys:
        'review_title', 'review_date_original',
        'review_rating', 'review_text', 'page_number'.
        Returns an empty list if no reviews are found.
    """
    data = []
    for i in range(from_page, to_page + 1):
        try:
            response = requests.get(url)  # Removed f-string as URL is passed as argument
            web_page = response.text
            soup = BeautifulSoup(web_page, "html.parser")

            for e in soup.select('article'):
                # Check if e.h2 exists before accessing its text attribute
              review_title = e.h2.text if e.h2 else None
              # Check if the element exists before accessing its text attribute
              date_element = e.select_one('[data-service-review-date-of-experience-typography]')
              review_date_original = date_element.text.split(': ')[-1] if date_element else None

              # Check if the rating element exists before accessing its 'alt' attribute
              rating_element = e.select_one('[data-service-review-rating] img')
              review_rating = rating_element.get('alt') if rating_element else None

              data.append({
                  'review_title': review_title,
                  'review_date_original': review_date_original,
                  'review_rating': review_rating,  # Use the review_rating variable here
                  'review_text': e.select_one('[data-service-review-text-typography]').text if e.select_one('[data-service-review-text-typography]') else None,
                  'page_number': i
              })

              logging.info(f"Scraped page {i} successfully.")
        except requests.exceptions.RequestException as e:
            logging.error(f"Error scraping page {i}: {e}")
        except Exception as e:
            logging.exception(f"Unexpected error scraping page {i}: {e}")

    return data

class TestScraping(unittest.TestCase):
    def test_scrape_reviews(self):
        """Tests the scrape_reviews function."""
        """Tests if scrape_reviews returns at least one review."""
        url = "https://uk.trustpilot.com/review/bt.com"
        from_page = 1
        to_page = 5  # Scrape 5 pages for testing
        reviews = scrape_reviews(url, from_page, to_page)
        self.assertTrue(len(reviews) > 0, "Should scrape at least one review")

if __name__ == "__main__":
    # Run the tests first
    unittest.main(argv=['first-arg-is-ignored'], exit=False)

    # Scrape reviews
    url = "https://uk.trustpilot.com/review/bt.com"
    from_page = 1
    to_page = 100
    data = scrape_reviews(url, from_page, to_page)

    # Save to CSV
    with open('zzz_my_result.csv', 'w', newline='') as output_file:
        dict_writer = csv.DictWriter(output_file, data[0].keys() if data else [])
        dict_writer.writeheader()
        dict_writer.writerows(data)

*This section was used as a testing ground for my web scraping to ensure it was able to pull the data.*

*The unit testing and logging variables had not been added at this stage.*

```
# This is formatted as code



from bs4 import BeautifulSoup
import requests, csv


data = []

from_page = 1
to_page = 15

for i in range(from_page, to_page + 1):
    response = requests.get(f"https://uk.trustpilot.com/review/bt.com")
    web_page = response.text
    soup = BeautifulSoup(web_page, "html.parser")

    for e in soup.select('article'):
        # Check if e.h2 exists before accessing its text attribute
        review_title = e.h2.text if e.h2 else None
        # Check if the element exists before accessing its text attribute
        date_element = e.select_one('[data-service-review-date-of-experience-typography]')
        review_date_original = date_element.text.split(': ')[-1] if date_element else None

        # Check if the rating element exists before accessing its 'alt' attribute
        rating_element = e.select_one('[data-service-review-rating] img')
        review_rating = rating_element.get('alt') if rating_element else None

        data.append({
            'review_title': review_title,
            'review_date_original': review_date_original,
            'review_rating': review_rating,  # Use the review_rating variable here
            'review_text': e.select_one('[data-service-review-text-typography]').text if e.select_one('[data-service-review-text-typography]') else None,
            'page_number': i
        })



with open('zzz_my_result.csv', 'w', newline='') as output_file:
    dict_writer = csv.DictWriter(output_file, data[0].keys())
    dict_writer.writeheader()
    dict_writer.writerows(data)
```

The next two code blocks assigns the pulled data to a variable 'df'.
This gives us a visual sanity check that we have actually pulled the correct daa into a structured format

In [ ]:
df=pd.read_csv('zzz_my_result.csv')

In [ ]:
df

# 2. Preprocessing and EDA

##Word Cloud Visualization of Customer Review Text
This function generate_and_display_wordcloud leverages the wordcloud and matplotlib libraries to provide a visual representation of the frequency distribution of words within customer review text data.

**Functionality**:

1. **Text Aggregation:** The function first aggregates all text from the 'review_text' column of the input Pandas DataFrame (df) into a single string (all_words). This ensures all review text contributes to the word cloud generation.

2. **Word Cloud Generation**: It utilizes the WordCloud class (from the wordcloud library) to generate the word cloud. Parameters like width, height, random_state, and max_font_size control the visualization's dimensions, reproducibility, and appearance. The generate() method of the WordCloud object processes the aggregated text (all_words) to create the word cloud.

3. **Visualization:** The function then uses matplotlib.pyplot (plt) to display the generated word cloud. plt.imshow renders the word cloud image, plt.axis('off') hides plot axes for a cleaner visual, plt.title adds a descriptive title, and plt.show presents the word cloud to the user.


**Purpose:**

The primary purpose is Exploratory Data Analysis (EDA). By visualizing word frequencies, data scientists can quickly gain insights into:

* **Dominant Themes:** Identifying the most prominent words in customer reviews can reveal recurring topics or themes.
* **Sentiment Clues:** While not a dedicated sentiment analysis tool, the word cloud can offer initial hints about customer sentiment based on the prevalence of positive or negative words.
* **Data Preprocessing Guidance:** The word cloud can inform subsequent text preprocessing steps, like identifying stop words for removal or terms for feature engineering.

**Libraries Used:**

* wordcloud: For generating the word cloud visualization.
* matplotlib.pyplot: For displaying the word cloud and customizing the plot.
* pandas: For data handling and manipulation (the input df is assumed to be a Pandas DataFrame).

**Assumptions:**

* Input DataFrame (df) contains a 'review_text' column with string-type data representing customer reviews.
* Necessary libraries (wordcloud, matplotlib.pyplot, pandas) are imported and available.

**Limitations:**

* Provides a high-level, frequency-based view; nuanced linguistic relationships or context are not captured.
* Interpretation requires domain knowledge and careful consideration of potential biases in the data.

In [ ]:
def generate_and_display_wordcloud(df):
    """Generates and displays a word cloud from review text in a Pandas DataFrame.

    This function takes a Pandas DataFrame containing review text and creates a word cloud visualization
    of the most frequent words in the reviews. It uses the 'review_text' column of the DataFrame
    as the source of text data.

    Args:
        df (pd.DataFrame): A Pandas DataFrame containing a 'review_text' column with review text data.

    Returns:
        None: This function displays the word cloud but does not return any value.
    """
    # To create a word cloud we'll extract all the words into one large list
    # then we input that into the WordCloud.generate() function to create our wordcloud plot
    all_words = ' '.join([text for text in df['review_text'].astype(str)])
    wordcloud = WordCloud(width=800, height=500, random_state=42, max_font_size=100).generate(all_words)

    plt.figure(figsize=(10, 7))
    plt.imshow(wordcloud, interpolation='bilinear')
    plt.axis('off')
    plt.title('Most Frequent Words in Reviews')
    plt.show()

In [ ]:
generate_and_display_wordcloud(df)

##Text Preprocessing for Customer Review Data

This code snippet performs a series of text cleaning and normalization operations on customer review data stored in a Pandas DataFrame (df). The objective is to prepare the text for downstream Natural Language Processing (NLP) tasks such as sentiment analysis or topic modeling.

**Functionality**:

1. **clean_text Function:**
This function encapsulates the text cleaning logic. It takes a single text string as input and applies the following transformations:

 * Removes HTML tags using regular expressions (re.sub(r'<.*?>', '', text)).
 * Removes URLs using regular expressions (re.sub(r'http\S+', '', text)).
 * Removes non-alphanumeric characters, retaining only letters and spaces (re.sub(r'[^a-zA-Z\s]', '', text)).
 * Converts the text to lowercase (text.lower()).
* The function includes input validation using isinstance() to ensure the input is a string.
* It returns the cleaned text string.
2. **Applying to DataFrame:**

* The apply() method is used to apply the clean_text function to the 'review_text' column of the DataFrame (df['review_text'].apply(clean_text)).
* The cleaned text is stored in a new column named 'cleaned_text' (df['cleaned_text'] = ...).

**Data Science Relevance:**

* **Data Cleaning:** The code removes noise and irrelevant elements (HTML tags, URLs, non-alphanumeric characters) from the raw text, improving the quality of data for analysis.
* **Text Normalization:** Lowercasing the text ensures case-insensitive processing, which is important for many NLP techniques.
* **Feature Engineering:** The cleaned text can be used as input for creating features for machine learning models (e.g., using TF-IDF or word embeddings).

**Advantages:**

* **Improved Data Quality:** The cleaning process enhances the quality and consistency of the text data.
* **Enhanced Model Performance:** Cleaned and normalized text can improve the accuracy and performance of NLP models.
* **Reduced Dimensionality:** Removing irrelevant characters and words can reduce the dimensionality of the data, making analysis more efficient.

**Considerations:**

* **Language Specificity:** The current cleaning process is tailored for English text. Adjustments might be needed for other languages.
* **Domain Adaptation:** Depending on the specific domain, additional cleaning steps might be necessary (e.g., removing specific jargon or abbreviations).
* **Preservation of Information:** While cleaning removes noise, it's essential to ensure that valuable information is not lost in the process.

In [ ]:
def clean_text(text):
    """
    Clean the input text by performing the following operations:
    1. Remove any HTML tags.
    2. Remove URLs.
    3. Remove non-alphanumeric characters.
    4. Convert the text to lowercase.

    Parameters:
    - text (str): The input string to be cleaned.

    Returns:
    - str: The cleaned version of the input text.
    """

    if isinstance(text,str):
    # Remove any HTML tags, URLs, and non-alphanumeric characters
      text = re.sub(r'<.*?>', '', text)
      text = re.sub(r'http\S+', '', text)
      text = re.sub(r'[^a-zA-Z\s]', '', text)
      # Convert text to lowercase
      text = text.lower()
    return text

df['cleaned_text'] = df['review_text'].apply(clean_text)
df


##Text Preprocessing Pipeline with NLTK for Sentiment Analysis and Topic Modeling

This code implements a standard text preprocessing pipeline using the Natural Language Toolkit (NLTK) library in Python. Its primary purpose is to prepare textual data, likely customer reviews in this context, for downstream tasks such as sentiment analysis and topic modeling.

**Functionality Breakdown:**
1. Stop Word Removal:

 * Utilizes nltk.corpus.stopwords to load a predefined list of common English words considered to have low semantic value.
 * Creates a set of stop words for efficient lookup during processing.

2. Stemming and Lemmatization:

 * Initializes a PorterStemmer for stemming, reducing words to their base form (e.g., "running" to "run").
 * Initializes a WordNetLemmatizer for lemmatization, a more nuanced approach
that considers word context to derive lemmas (e.g., "better" to "good").

3. process_text Function:

 * Encapsulates the core preprocessing logic.
 * Includes input validation to ensure the input is a string.
 * Splits the input text into individual words using text.split().
 * Iterates through words, removing stop words using list comprehension.
 * Applies stemming and then lemmatization to each remaining word.
 * Joins the processed words back into a single string.

4. DataFrame Integration:

 * Assumes a Pandas DataFrame (df) with a 'cleaned_text' column containing pre-cleaned text.
 * Applies the process_text function to the 'cleaned_text' column using apply().
 * Stores the processed text in a new 'processed_text' column in the DataFrame.
 * Optionally displays the first few rows using df.head() for inspection.

**Data Science Relevance:**

* Noise Reduction: Removes stop words and irrelevant characters, enhancing data quality.
* Dimensionality Reduction: Stemming and lemmatization reduce vocabulary size, improving model efficiency.
* Feature Engineering: Creates a 'processed_text' feature for NLP models, optimized for sentiment analysis and topic modeling.
* Improved Model Performance: Cleaned and standardized text can lead to more accurate and robust model outcomes.

**Considerations:**
* Language Specificity: The code assumes English text. Adaptations may be needed for other languages.
* Domain Adaptation: Consider additional preprocessing steps (e.g., removing domain-specific jargon) if necessary.
* Stemming vs. Lemmatization: Stemming is faster but can produce non-words.
Lemmatization is more accurate but computationally expensive. Choose the
appropriate method based on the specific use case.

**Advantages:**
* Modular and Reusable: The process_text function can be easily applied to other text data.
* Standard Pipeline: Follows established preprocessing practices for NLP tasks.
* Efficient Implementation: Utilizes optimized data structures (sets) and list comprehensions for performance

In [ ]:
# Load NLTK stopwords

stop_words = set(stopwords.words('english'))

# Initialising stemmer and lemmatiser
stemmer = PorterStemmer()
lemmatiser = WordNetLemmatizer()

def process_text(text):
  """Processes text by removing stop words, stemming, and lemmatization.

    This function takes a string of text as input, performs the following steps:
    1. Splits the text into individual words.
    2. Removes common English stop words (e.g., "the", "a", "is").
    3. Applies stemming to reduce words to their root form (e.g., "running" to "run").
    4. Applies lemmatization to find the base form of words (e.g., "better" to "good").
    5. Joins the processed words back into a single string.

    If the input is not a string, it returns an empty string.

    Args:
        text (str): The text to be processed.

    Returns:
        str: The processed text.
    """
  if isinstance(text,str):
    words = text.split()
    words = [word for word in words if word not in stop_words] # Remove stopwords
    words = [stemmer.stem(word) for word in words] # Stemming
    words = [lemmatiser.lemmatize(word) for word in words] # Lemmatisation
    return ' '.join(words)
  else:
    return ''

df['processed_text'] = df['cleaned_text'].apply(process_text)
df.head()

*This code was leftover in the process, when encoding words based on the level of importance using the TfidfVectorizer was considered.*

*This method was abandoned as the goal of the project had changed and better packages that was more in line with fuliflling the objectives was used instead.*



```
"""
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

tfidf_vectoriser = TfidfVectorizer(max_features=5000)
X = tfidf_vectoriser.fit_transform(df['processed_text']).toarray()
"""
```




# **3. Preliminary sentiment analysis**

# Sentiment Analysis using VADER

This function `get_sentiment` performs sentiment analysis on text input using the VADER (Valence Aware Dictionary and Sentiment Reasoner) lexicon from NLTK.

**Functionality:**

- Takes a text string as input.
- Uses VADER to calculate sentiment scores.
- Classifies the sentiment as either positive (1) or negative (0) based on the positive sentiment score.
- Returns the binary sentiment classification (1 or 0).

**Purpose:**

This function is designed for quick and efficient sentiment analysis, particularly useful for tasks involving social media text, informal language, and customer feedback.

**Implementation:**

1. **Initialization:** Creates a VADER sentiment analyzer instance analyzer.
2. **Input Validation:** Checks if the input `text` is a string. Raises a TypeError if it's not.
3. **Sentiment Calculation:**
    - Uses analyzer.polarity_scores(text) to get sentiment scores.
    - Assigns a binary sentiment score (1 for positive, 0 for negative) based on the positive score (scores['pos']).
4. **Return:** Returns the binary sentiment classification.

**Advantages:**

- Simple to use.
- Fast and efficient.
- Well-suited for social media and informal text.

**Limitations:**

- Provides a binary classification, potentially missing nuanced sentiment.
- Context-dependent; may not accurately interpret complex language features.

**Libraries Used:**

- **NLTK:** The Natural Language Toolkit, used for text processing and sentiment analysis.  (Make sure to `import nltk` and download the VADER lexicon: `nltk.download('vader_lexicon')`)


In [ ]:

analyzer = SentimentIntensityAnalyzer()

def get_sentiment(text):
    """Analyzes the sentiment of a text using VADER and returns a binary classification.

    This function utilizes the VADER (Valence Aware Dictionary and sEntiment Reasoner)
    lexicon to determine the sentiment expressed in the input text. It returns 1
    if the text is classified as positive and 0 if it is classified as negative.

    Args:
        text (str): The input text to be analyzed for sentiment.

    Returns:
        int: 1 if the sentiment is positive, 0 if negative.

    Raises:
        TypeError: if the input 'text' is not a string.

    Example:
        >>> get_sentiment("This is a great product!")
        1
        >>> get_sentiment("I am very disappointed with this service.")
        0
    """

    if not isinstance(text, str):
        raise TypeError("Input 'text' must be a string.")

    scores = analyzer.polarity_scores(text)

    sentiment = 1 if scores['pos'] > 0 else 0

    return sentiment

In [ ]:
df['sentiment'] = df['processed_text'].apply(get_sentiment)


In [ ]:
df['sentiment'].value_counts()



---



---



#4. BERT Sentiment Analysis

This section utilizes a pre-trained BERT (Bidirectional Encoder Representations from Transformers) model for sentiment analysis on customer reviews. BERT is a state-of-the-art NLP model known for its ability to understand context and nuances in text, potentially leading to more accurate sentiment predictions.
This is a more robust alternative to sentiment analysis then using the Vader lexicon. BERT's advanced contextual understanding and adaptability often make it superior for sentiment analysis, especially in complex and varied datasets. However, VADER's simplicity and speed can be advantageous in specific scenarios.
(Saha et al., 2023)

Model Loading and Preparation

1.  **Loading the Model**: The code model = BertForSequenceClassification.from_pretrained() loads a pre-trained BERT model designed for sentiment classification. The model is downloaded from the specified repository ("nlptown/bert-base-multilingual-uncased-sentiment").
2.  **Setting to Evaluation Mode**: model.eval() sets the model to evaluation mode, which is crucial for inference. It disables training-specific behaviors and ensures consistent predictions.
3.  **Loading the Tokenizer**: tokenizer = BertTokenizer.from_pretrained() loads the tokenizer associated with the BERT model. The tokenizer converts raw text into numerical representations (tokens) that the model can understand.

Sentiment Prediction Function (`get_bert_sentiment`)

This function takes text input and predicts its sentiment using the BERT model:

1.  **Tokenization and Input Formatting**: Uses the tokenizer to convert the input `text` into a format suitable for BERT, including padding and truncation to handle varying text lengths.
2.  **Model Inference**: Feeds the prepared input to the BERT model to obtain predictions (logits).
3.  **Probability Calculation**: Converts the raw logits into probabilities using the softmax function, representing the likelihood of each sentiment label.
4.  **Sentiment Classification**: Determines the predicted sentiment label by selecting the class with the highest probability.
5.  **Label Mapping**: Maps the numerical class prediction to human-readable sentiment labels (e.g., "Positive", "Negative").

Applying BERT to the DataFrame:

The line `df['bert_sentiment'] = df['review_text'].astype(str).apply(get_bert_sentiment)` applies the `get_bert_sentiment` function to the 'review_text' column of the DataFrame (`df`), creating a new column named 'bert_sentiment' to store the predicted sentiment labels for each review.


Data Science Relevance:

 *   **Contextual Understanding**: BERT's bidirectional architecture enables it to consider the context of words, improving accuracy, especially for complex language.
 *   **State-of-the-Art Performance**: BERT has achieved state-of-the-art results on various NLP tasks, including sentiment analysis.
 *   **Pre-trained and Ready to Use**: Pre-trained BERT models reduce the need for extensive training data and resources.

Libraries Used:

*   **transformers:** Provides the `BertForSequenceClassification` and `BertTokenizer` classes for loading and using the BERT model.
*   **torch:** The PyTorch library, used for tensor operations and working with the BERT model.


In [ ]:
model = BertForSequenceClassification.from_pretrained("nlptown/bert-base-multilingual-uncased-sentiment")
model.eval()
tokenizer = BertTokenizer.from_pretrained("nlptown/bert-base-multilingual-uncased-sentiment")

def get_bert_sentiment(text):
    """Analyzes the sentiment of a text using a pre-trained BERT model.

    This function utilizes a pre-trained BERT model to classify the sentiment
    expressed in the input text. It returns the sentiment as a categorical label:
    "Very Negative", "Negative", "Neutral", "Positive", or "Very Positive".

    Args:
        text (str): The input text to be analyzed for sentiment.

    Returns:
        str: The sentiment label predicted by the BERT model.
              Possible values: "Very Negative", "Negative", "Neutral",
                              "Positive", "Very Positive", "Unknown".
    """
    inputs = tokenizer(text, padding=True, truncation=True, return_tensors="pt")
    outputs = model(**inputs)
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)
    predicted_class = torch.argmax(probabilities, dim=-1).item()

    # Map predictions to sentiment labels
    sentiment_map = {
          1: "Very Negative",
          2: "Negative",
          3: "Neutral",
          4: "Positive",
          5: "Very Positive"
      }
    return sentiment_map.get(predicted_class + 1, "Unknown")  # Returns sentiment as a label

df['bert_sentiment'] = df['review_text'].astype(str).apply(get_bert_sentiment)

# 4.Exploring and critical analysis of Sentiment Data

##Visualising sentiment counts

In this section, I am breaking down the counts of the sentiments outputted by the BERT sentiment analysis. At the time of writing, there was only Very Negative and Negative reviews. This is is subject to change depending on the data pulled from the webscraping script.

Libaries used:
* Matplotlib was used to generate the bar charts to visualise the counts

In [ ]:
df["bert_sentiment"].value_counts().plot(kind='bar', color = ['red','orange','yellow','purple','green'])
plt.show()


In [ ]:
df

##Topic Modeling and Hierarchical Clustering with BERTopic

This code focuses on using the BERTopic library for topic modeling and hierarchical clustering. Its purpose is to identify underlying topics within a collection of text documents and then group similar topics together to reveal higher-level themes or relationships. (Xie, Liu and Wang, 2017; Mohamad, Sulaiman and Kamaruddin, 2018)

**Functionality Breakdown:**

1. **Data Preparation:**

 * docs = df['processed_text'].tolist(): Extracts the preprocessed text data from the 'processed_text' column of a Pandas DataFrame (df) and converts it into a Python list called docs. This list serves as the input for topic modeling.
2. **BERTopic Model Creation:**

* topic_model = BERTopic(language="english", calculate_probabilities=True, n_gram_range=(1, 2), nr_topics=20): Initializes a BERTopic model with specific configurations:
 * language="english": Specifies the language of the text data.
 * calculate_probabilities=True: Enables the calculation of probabilities for document-topic assignments.
 * n_gram_range=(1, 2): Considers both individual words (1-grams) and two-word combinations (2-grams) for topic extraction.
 * nr_topics=20: Sets the target number of topics to be discovered (adjustable).

3. **Model Fitting and Transformation:**

 * topics, probs = topic_model.fit_transform(docs): Applies the BERTopic model to the input text data (docs).
 * fit: Trains the model to learn topics from the text.
 * transform: Assigns each document to the most relevant topic(s).
 * The results are stored in two variables: topics (topic assignments for each document) and probs (probabilities of document-topic assignments).

4. **Hierarchical Clustering:**

 * Similarity Calculation:
  similarity_matrix = cosine_similarity(topic_model.topic_embeddings_): Calculates the cosine similarity between the embeddings of different topics, representing their semantic relatedness.
 * Clustering:
linkage_matrix = linkage(similarity_matrix, method='ward'): Performs hierarchical clustering using the 'ward' method on the similarity_matrix to group similar topics together. This creates a hierarchical structure represented by linkage_matrix.

**Data Science Relevance:**

* Topic Discovery: Identifies the main themes and subjects discussed in the text data.
* Clustering Similar Topics: Groups related topics to uncover higher-level themes and relationships.
Document Organization: Categorizes documents based on their topic assignments, facilitating analysis and exploration.
* Insight Generation: Provides a deeper understanding of the content and structure of text data.

**Advantages:**

* Unsupervised Learning: Automatically discovers topics without the need for labeled data.
* Hierarchical Structure: Reveals topic relationships through clustering.  (Chaturvedi, Shankar and Srivastava, 2018; Wang, Wang and Zhang, 2019).


* Interpretability: Offers insights into the content of each topic.
Flexibility: Allows for customization of parameters to suit specific datasets.

**Libraries Used:**

* bertopic: For topic modeling and hierarchical clustering.
* sklearn: For cosine similarity calculation and hierarchical clustering (using cosine_similarity and linkage from sklearn.metrics.pairwise and scipy.cluster.hierarchy, respectively).

**Considerations:**

* Parameter Tuning: The number of topics (nr_topics) and other parameters might need adjustment depending on the dataset and desired granularity of topics.
* Interpretation of Topics: Careful examination of the most representative words for each topic is crucial for accurate interpretation.
* Data Preprocessing: The quality of topic modeling results depends on the effectiveness of prior text preprocessing steps.

In [ ]:
docs = df['processed_text'].tolist()


# Create a BERTopic model
topic_model = BERTopic(language="english", calculate_probabilities=True, n_gram_range=(1, 2), nr_topics=20)  # Adjust nr_topics


topics, probs = topic_model.fit_transform(docs)


# Calculate the similarity matrix instead of distance matrix
similarity_matrix = cosine_similarity(topic_model.topic_embeddings_)

# Perform hierarchical clustering
linkage_matrix = linkage(similarity_matrix, method='ward')




In [ ]:

topic_model.visualize_hierarchy(linkage_function=lambda x: linkage_matrix)


In [ ]:
topic_model.visualize_barchart()

## Generating Word Clouds for Specific Topics

This function `create_topic_wordcloud` generates and displays a word cloud visualization
for a specific topic extracted by a topic modeling process, likely using BERTopic.
It aims to provide a visual representation of the prominent words within a given topic.

**Functionality:**

1. **Topic Word Retrieval**:
    - `topic_words = topic_model.get_topic(topic_num)`: This line retrieves the words
      associated with the specified `topic_num` from the `topic_model` object. It
      likely returns a list of (word, frequency/importance) tuples for the topic.

2. **Word Cloud Generation**:
    - `wordcloud = WordCloud(...).generate_from_frequencies(...)`: This creates a
      WordCloud object using the `WordCloud` class from the `wordcloud` library.
      It configures the word cloud's appearance (width, height, background color,
      minimum font size). The `generate_from_frequencies()` method is used to
      generate the word cloud based on the frequencies of words within the topic,
      making more frequent words appear larger in the visualization.

3. **Word Cloud Display**:
    - `plt.figure(...)`, `plt.imshow(...)`, `plt.axis(...)`, `plt.tight_layout(...)`,
      `plt.title(...)`, `plt.show()`: These lines use Matplotlib to display the
      generated word cloud. They create a figure, display the word cloud image,
      hide the axes, adjust spacing, set the title indicating the topic number,
      and finally, show the plot to the user.

**Data Science Relevance:**

* **Topic Interpretation:** Word clouds provide a visual and intuitive way to
  understand the main themes or concepts captured by a topic.
* **Exploratory Data Analysis (EDA)**:  They are useful for exploring topic modeling
  results and identifying the most relevant words within each topic.
* **Communication:** Word clouds can effectively communicate topic insights to
  stakeholders or audiences who might not be familiar with the underlying data
  or modeling techniques.

**Libraries Used:**

* **wordcloud:** For generating word cloud visualizations.
* **matplotlib.pyplot:** For displaying the word cloud and customizing the plot.


In [ ]:
def create_topic_wordcloud(topic_model, topic_num):
    """Generates a word cloud for a specific topic.
    """
    topic_words = topic_model.get_topic(topic_num)
    wordcloud = WordCloud(width=800, height=500,
                          background_color='white',
                          min_font_size=10).generate_from_frequencies({word[0]: word[1] for word in topic_words})

    plt.figure(figsize=(8, 8), facecolor=None)
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.tight_layout(pad=0)
    plt.title(f"Word Cloud for Topic {topic_num}")
    plt.show()


topic_num = 2
create_topic_wordcloud(topic_model, topic_num)

##Generating a Sentiment-Highlighted Word Cloud for a Specific Topic

* Red: Negative sentiment
* Yellow: Neutral sentiment
* Green: Positive sentiment

The color of each word in the word cloud will be based on its sentiment score.

From this point on, this is key information that can be used by customer agents and to aserctain the most severe customer pain points which will be highlighted in the red colour.

This function `create_topic_wordcloud` generates and displays a word cloud visualization
for a specific topic, highlighting words based on their sentiment (positive, negative, or neutral).

**Functionality:**

1. Sentiment Analyzer Initialization:
    - `analyzer = SentimentIntensityAnalyzer()`: Initializes a VADER sentiment analyzer to calculate word sentiments.

2. `create_topic_wordcloud` Function:
    - Takes a `topic_model` (e.g., from BERTopic) and a `topic_num` as input.
    - Retrieves topic words using `topic_model.get_topic(topic_num)`.
    - Calculates sentiment scores for each word using VADER's `polarity_scores()`.
    - Generates a word cloud using `WordCloud`, applying the 'RdYlGn' colormap to highlight sentiment:
        - Red: Negative sentiment
        - Yellow: Neutral sentiment
        - Green: Positive sentiment
    - Displays the word cloud using Matplotlib, including title and layout adjustments.

**Data Science Relevance:**

* Sentiment-Aware Topic Interpretation: Provides a visual representation of topic words with sentiment highlighting, aiding in understanding the emotional tone associated with each topic.
* Enhanced Exploratory Data Analysis (EDA): Allows for a deeper exploration of topic modeling results by incorporating sentiment information.
* Actionable Insights: Identifying negative sentiment words within topics can help pinpoint customer pain points or areas needing attention.

**Libraries Used:**

* nltk.sentiment.vader: For sentiment analysis using VADER.
* wordcloud: For generating word cloud visualizations.
* matplotlib.pyplot: For displaying the word cloud.


In [ ]:
analyzer = SentimentIntensityAnalyzer()

def create_topic_wordcloud(topic_model, topic_num):
    """Generates a word cloud for a specific topic, highlighting sentiment words.
    """
    topic_words = topic_model.get_topic(topic_num)

    # Create a dictionary mapping words to sentiment scores
    word_sentiments = {word[0]: analyzer.polarity_scores(word[0])['compound'] for word in topic_words}

    # Generate word cloud, using sentiment scores for color mapping
    wordcloud = WordCloud(width=800, height=500,
                          background_color='white',
                          min_font_size=10,
                          colormap='RdYlGn'  # Use a diverging colormap
                         ).generate_from_frequencies({word[0]: word[1] for word in topic_words})


    plt.figure(figsize=(8, 8), facecolor=None)
    plt.imshow(wordcloud)
    plt.axis("off")
    plt.tight_layout(pad=0)
    plt.title(f"Word Cloud for Topic {topic_num}")
    plt.show()

topic_num = 4
create_topic_wordcloud(topic_model, topic_num)

*This is abandoned code that would show the distribution of sentiment within the topics as a box plot graph. Due to the data being heavily skewed towards negative, it felt unnecessary to visualise this data as it would not have given enough significant insight into the data itself*


```
df['topic'] = topics

# Create a box plot
plt.figure(figsize=(12, 6))
sns.boxplot(x='topic', y='sentiment', data=df)
plt.title('Sentiment Distribution within Topics (Box Plot)')
plt.xlabel('Topic')
plt.ylabel('Sentiment Score')
plt.xticks(rotation=45, ha='right')  # Rotate x-axis labels for better readability
plt.show()

# Create a violin plot
plt.figure(figsize=(12, 6))
sns.violinplot(x='topic', y='sentiment', data=df)
plt.title('Sentiment Distribution within Topics (Violin Plot)')
plt.xlabel('Topic')
plt.ylabel('Sentiment Score')
plt.xticks(rotation=45, ha='right')
plt.show()

topic_info_df = topic_model.get_topic_info()
topic_labels = dict(zip(topic_info_df.Topic, topic_info_df.Name))
plt.figure(figsize=(12, 6))
sns.boxplot(x='topic', y='sentiment', data=df)
plt.title('Sentiment Distribution within Topics (Box Plot)')
plt.xlabel('Topic')
plt.ylabel('Sentiment Score')
plt.xticks(ticks=topic_info_df.Topic, labels=topic_info_df.Name, rotation=45, ha='right')  
plt.show()
```



*In this abandoned code block, an attempt was made to gain deeper insight into topics that a high count of co-occurence with each other and attempting to visualise this. Currently the number of topics visualised is too messy to be considered useful exploratory knowledge.*

*This has been left in as with further tuning and testing, this can be an especially useful insight into grouping common pain points with customers.*
```
topic_co_occurrence = defaultdict(lambda: defaultdict(int))

for row in df.itertuples():
    # Get the topic for the current review
    topic = row.topic
    
    # If the topic is not -1 (outlier), count co-occurrences
    if topic != -1:  
        # Iterate through all other reviews to check for co-occurrence
        for other_row in df.itertuples():  
            other_topic = other_row.topic
            if other_topic != -1 and row.Index != other_row.Index:
                topic_co_occurrence[topic][other_topic] += 1

# Print or analyze the topic_co_occurrence dictionary
for topic1, co_occurring_topics in topic_co_occurrence.items():
    for topic2, count in co_occurring_topics.items():
        if count > 10:  # Set a threshold for frequent co-occurrence
            print(f"Topic {topic1} frequently co-occurs with Topic {topic2} (count: {count})")
            
import networkx as nx
import matplotlib.pyplot as plt

# Create a networkx graph
graph = nx.Graph()

# Add nodes (topics) to the graph
for topic in topic_co_occurrence:
    graph.add_node(topic)

# Add edges (co-occurrences) to the graph
for topic1, co_occurring_topics in topic_co_occurrence.items():
    for topic2, count in co_occurring_topics.items():
        if count > 10:  # Adjust threshold as needed
            graph.add_edge(topic1, topic2, weight=count)

# Visualize the graph
plt.figure(figsize=(10, 8))
pos = nx.spring_layout(graph, k=0.15, seed=42)  # Adjust layout parameters
nx.draw(graph, with_labels=True, labels=topic_labels, node_color='skyblue', node_size=1500,
        edge_color='gray', font_size=10, pos=pos)  
plt.title("Topic Co-occurrence Network")
plt.show()
```



*In this abandoned code segment, training the data on various supervised learning models was still being considered. This ended up being outside the scope of the probject and other alternative methods was considered on acheiving high accuracy in sentiment analysis*



```

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

# Logistic Regression
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

# Random Forest
rf = RandomForestClassifier()
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

# SVM
svm = SVC()
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report

# Logistic Regression
print("Logistic Regression Performance:")
print(accuracy_score(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

# Random Forest
print("\nRandom Forest Performance:")
print(accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

# SVM
print("\nSVM Performance:")
print(accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))
```



##LLM Usage:

LLM has been used on debugging various code sections throughout this notebook

There has been such limited positive data to train the BERT model on, making it difficult to aserctain customer sentiment.

AI has been used to fine tune functions to help with making the code more concise and readable. This is to aid readability and repeatability by other data scientists.
Rewriting and making notes more concise in the markdown comments has been another use of AI.
A technique to deal with class imbalance is undersampling the negative reviews, to be able to train the models properly on the data


#References

Richardson, L., 2007. Beautiful Soup Documentation. [online] Available at: https://www.crummy.com/software/BeautifulSoup/bs4/doc/ [Accessed 7 May 2025].

Saha, S., Imran, M., Md. Motinur Rahman and Hasan, Z. (2023). VADER vs. BERT: A Comparative Performance Analysis for Sentiment on Coronavirus Outbreak. pp.371–385. doi: https://doi.org/10.1007/978-3-031-34619-4_30.

Xie, S., Liu, W., and Wang, D., 2017. A hybrid approach to topic modelling and sentiment analysis for customer review data. Expert Systems with Applications, 71, pp.123–132. https://doi.org/10.1016/j.eswa.2016.11.048

Mohamad, A., Sulaiman, M.N. and Kamaruddin, S., 2018. Topic modelling and sentiment analysis of reviews to improve customer satisfaction in the retail industry. International Journal of Advanced Computer Science and Applications, 9(6), pp.87–95. https://doi.org/10.14569/IJACSA.2018.090614

Chaturvedi, A., Shankar, R., and Srivastava, S., 2018. Hierarchical clustering-based topic modelling and sentiment analysis for product review data. International Journal of Computer Applications, 179(23), pp.16–21. https://doi.org/10.5120/ijca2018916503

Wang, W., Wang, C. and Zhang, X., 2019. Sentiment analysis and hierarchical clustering for product review analysis. International Journal of Computational Intelligence Systems, 12(6), pp.1074–1084. https://doi.org/10.2991/ijcis.d.191113.001
